# Evals and Observability

**What you will build:** a small **evaluation suite** that scores your agent on a set of cases, plus one-line **observability**. This is new versus the n8n course — and it's what separates a demo from something you can trust in production. It follows directly from 0.0's hard truth: LLM output is non-deterministic, so you measure quality, you don't eyeball it.

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/16_evals_and_observability.ipynb) (nothing to install), or run locally in Jupyter. Each notebook installs what it needs in its first cell.

In [ ]:
# Setup — PydanticAI on OpenRouter. Run once.
!pip install -q "pydantic-ai-slim[openai]>=2.0,<3.0" "pydantic-evals>=2.0,<3.0"   # verified against pydantic-ai 2.5 (2026)

import os, getpass
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Paste your OpenRouter API key: ")

MODEL_NAME = "meta-llama/llama-3.3-70b-instruct"
model = OpenAIChatModel(
    MODEL_NAME,
    provider=OpenAIProvider(base_url="https://openrouter.ai/api/v1",
                            api_key=os.environ["OPENROUTER_API_KEY"]),
)
print("Ready:", MODEL_NAME)

## Why evals

You changed a prompt and it "seems better." Better on what? On how many cases? An eval turns that gut feeling into a number you can track as you change models and prompts. The workflow: define **cases** (input + what you expect), pick **evaluators** (how to score), run the **dataset** against your agent.

## A minimal eval suite

We'll evaluate a small classification agent.

In [ ]:
from typing import Literal
from pydantic_evals import Case, Dataset
from pydantic_evals.evaluators import EqualsExpected

# output_type=Literal makes the output a checked value (1.3) — no .strip().lower(),
# no "Spam." with a full stop sneaking past the comparison:
classifier = Agent(model, output_type=Literal["spam", "ham"],
                   system_prompt="Classify the message as spam or ham.")

async def classify(text: str) -> str:
    return (await classifier.run(text)).output

dataset = Dataset(
    name="spam-detector",
    cases=[
        Case(name="promo",  inputs="WIN a FREE iPhone now!!! click here", expected_output="spam"),
        Case(name="normal", inputs="Are we still on for lunch tomorrow?", expected_output="ham"),
        Case(name="invoice", inputs="Your invoice for March is attached.", expected_output="ham"),
    ],
    evaluators=[EqualsExpected()],   # compares the output against each case's expected_output
)

report = await dataset.evaluate(classify)
report.print(include_input=True, include_output=True)

That table is your ground truth. Change the model in the setup cell, rerun this one cell, and you can *see* whether the cheaper model still passes — instead of hoping. This is how you compare models responsibly.

## Error analysis: the loop that makes evals a tool, not a ritual

A score is only useful if you act on it. The real workflow is a loop: run the eval, look at what **failed**, find the cause, fix it, re-run. Let's add a case the current classifier gets wrong.

In [ ]:
hard = Dataset(
    name="hard-cases",
    cases=[Case(name="newsletter", inputs="Your weekly product newsletter -- 3 new features inside.", expected_output="ham")],
    evaluators=[EqualsExpected()],
)
report = await hard.evaluate(classify)
report.print(include_input=True, include_output=True)   # likely FAILS: 'newsletter' looks promotional -> 'spam'

The failure is informative: the model treats anything promotional as spam, but a newsletter the user subscribed to is *ham*. The eval didn't just hand us a number — it pointed at a missing distinction in the prompt. Fix the cause (the system prompt), then re-run the same case.

In [ ]:
classifier2 = Agent(model, output_type=Literal["spam", "ham"], system_prompt=(
    "Classify the message as 'spam' (unsolicited bulk or junk) or 'ham' (wanted mail, "
    "INCLUDING newsletters and updates the user signed up for)."))

async def classify2(text: str) -> str:
    return (await classifier2.run(text)).output

report = await hard.evaluate(classify2)
report.print(include_input=True, include_output=True)   # now PASSES

That's error-driven development: the failing case pointed at the fix, and the eval confirmed it worked. Do this in a loop — add a case every time the agent surprises you — and your eval set becomes a growing spec of what "working" means. This loop, not the score itself, is what separates evals-as-engineering from evals-as-ceremony.

## Custom evaluator: LLM as judge

Built-in evaluators (`Contains`, `EqualsExpected`, `IsInstance`) cover objective checks. For subjective quality — "is the tone empathetic?" — you write your own by subclassing `Evaluator`. Here's one that uses an LLM as the judge: the same idea as the `judge()` function in notebook 0.5, now wired into the test harness so it runs over a whole dataset.

In [ ]:
from pydantic_evals.evaluators import Evaluator, EvaluatorContext

judge_agent = Agent(model, system_prompt="You are a strict, fair reviewer.")

class ToneJudge(Evaluator[str]):
    """Use an LLM to judge whether the output sounds professional and empathetic."""
    async def evaluate(self, ctx: EvaluatorContext[str]) -> float:
        verdict = await judge_agent.run(
            f"Does this response sound professional and empathetic? Answer 'yes' or 'no'.\n\n"
            f"Response: {ctx.output}")
        return 1.0 if "yes" in verdict.output.lower() else 0.0

# Evaluate a support-reply agent with the LLM judge:
reply_agent = Agent(model, system_prompt="You write brief, professional customer-support replies.")

async def write_reply(complaint: str) -> str:
    r = await reply_agent.run(complaint)
    return r.output

tone_dataset = Dataset(
    name="tone-eval",
    cases=[Case(name="angry", inputs="Your app deleted my data and I am furious!")],
    evaluators=[ToneJudge()],
)
report = await tone_dataset.evaluate(write_reply)
report.print(include_input=True, include_output=True)

```{note}
This is the same code-rule vs LLM-judge hierarchy from notebook 0.5, now applied to automated testing. Prefer code checks when the rule is objective; reach for an LLM judge only for subjective quality. (pydantic-evals also ships a built-in `LLMJudge(rubric=...)` — we hand-rolled one here to show the `Evaluator` mechanism, but reach for the built-in in real use.) See the [pydantic-evals docs](https://ai.pydantic.dev/evals/).
```

## Evaluating an agent with tools: score the *path*, not just the answer

Everything above evaluated a bare classifier. Real agents call tools — and a right answer reached the *wrong way* is a failure you'll pay for later: an agent that answers a stock question from its training data instead of the price tool is correct today and stale tomorrow. So evals for agents check the **trajectory** too: *did it call the tool it was supposed to?*

The trick is to make the eval task return both the answer *and* the trace, then write evaluators for each:

In [ ]:
from dataclasses import dataclass

stock_agent = Agent(model, system_prompt="You are a helpful assistant. Use tools when they help.")

@stock_agent.tool_plain
def get_stock_price(ticker: str) -> str:
    """Get the latest price for a stock ticker symbol, e.g. 'AAPL'."""
    return {"AAPL": "224.10", "GOOGL": "178.30"}.get(ticker.upper(), "unknown")

async def answer_with_trace(question: str) -> dict:
    r = await stock_agent.run(question)
    return {"answer": r.output,
            "tools": [p.tool_name for m in r.all_messages()
                      for p in m.parts if p.part_kind == "tool-call"]}

@dataclass
class UsedTool(Evaluator[str, dict]):          # evaluators with fields must be dataclasses
    tool_name: str
    def evaluate(self, ctx: EvaluatorContext[str, dict]) -> bool:
        return self.tool_name in ctx.output["tools"]

@dataclass
class AnswerContains(Evaluator[str, dict]):
    def evaluate(self, ctx: EvaluatorContext[str, dict]) -> bool:
        return str(ctx.expected_output) in ctx.output["answer"]

stock_eval = Dataset(
    name="stock-agent",
    cases=[
        Case(name="aapl",  inputs="What is the price of AAPL stock?",  expected_output="224.10"),
        Case(name="googl", inputs="What's GOOGL trading at right now?", expected_output="178.30"),
    ],
    evaluators=[UsedTool(tool_name="get_stock_price"), AnswerContains()],
)

report = await stock_eval.evaluate(answer_with_trace)
report.print(include_input=True, include_output=True)

Each case now carries **two verdicts**: the answer is right *and* it came through the tool. If the model ever answers from memory, `AnswerContains` may still pass (stale-but-plausible prices are exactly how this fails) while `UsedTool` catches it — that's the point of scoring the path. This pattern scales to everything you've built: "did the RAG agent search before answering?" (1.7), "did the refund stay under the cap?" (1.5). Industry courses call these **trajectory evals**; you now have the mechanism.

::::{dropdown} 🛠️ Try it yourself
:color: secondary

1. **Add a case that fails.** Insert a spam message labelled `"ham"` into the first dataset and watch the report catch it. **Done when:** you can explain why a deliberately wrong label is a useful test of the *eval itself*.
2. **Write a length evaluator.** Create a custom `Evaluator` that scores `1.0` when the output is under 120 characters, and add it to the tone dataset. **Done when:** both ToneJudge and your length check appear per case in the report.
3. **Compare models.** Run `stock_eval` with two different `MODEL_NAME` values and compare pass rates — especially `UsedTool`. **Done when:** you have a table of model vs pass-rate, which is how model choices should be made.
::::

::::{dropdown} 🛠️ Solutions
:color: secondary

**1 —** The report flags the mislabeled case as a failure — which is *the eval working*: it can only be as good as its labels, and a poisoned label shows up as a persistent, confusing failure. When a case keeps failing after several honest fixes, audit the label before the agent.

**2 —**

```python
@dataclass
class MaxLength(Evaluator[str, str]):
    limit: int = 120
    def evaluate(self, ctx: EvaluatorContext[str, str]) -> bool:
        return len(ctx.output) <= self.limit

tone_dataset = Dataset(name="tone-eval",
    cases=[Case(name="angry", inputs="Your app deleted my data and I am furious!")],
    evaluators=[ToneJudge(), MaxLength(limit=120)])
```

Code check + LLM judge on the same dataset — the 0.5 hierarchy, mechanized.

**3 —** Change `MODEL_NAME`, re-run setup + the `stock_eval` cell, note the pass rates. Small models typically keep `AnswerContains` alive longer than `UsedTool` — they answer plausibly while skipping tools, the exact failure mode trajectory evals exist to catch.
::::

## Observability in one line

When an agent misbehaves you need to see every model call, tool call, and token. **Logfire** (from the Pydantic team) instruments PydanticAI with one call. It's optional and needs a free account, so this cell is illustrative — uncomment to use it.

In [ ]:
# !pip install -q "pydantic-ai-slim[logfire]"
# import logfire
# logfire.configure()               # prompts for a free token on first run
# logfire.instrument_pydantic_ai()  # now every agent.run(...) is traced
print("Observability is optional. Uncomment above to trace runs in Logfire.")

```{note}
In n8n you watched execution on the canvas. In code, observability is something you instrument — Logfire here, or LangSmith in the LangGraph world. Same purpose: see what the agent actually did.
```

**What's next:** the milestone of Block 1 — **1.7**, the **knowledge agent**: an agent that answers from *your* documents (RAG).